#### Install

In [ ]:
!pip install -U langchain openai langchain-openai langchain-chroma langchain-experimental -qqq
!pip install --upgrade nltk -qqq
!sudo apt-get install tesseract-ocr
!apt-get install -y poppler-utils
!pip install "unstructured[all-docs]" pillow pydantic lxml pillow matplotlib chromadb tiktoken
!pip install camelot-py
!pip install PyMuPDF
!pip install pymupdf4llm
!pip install PyPDF2
!pip install pdfminer.six

### Обработка данных

#### Видео и аудио

stable_whisper позволяет транскрибировать видео, аудио и сохранить как удобно в txt, ass (расширение длч субтитров), с разным уровнем разделения (по словам, предложениям, полный текст)

In [ ]:
!pip install -U stable-ts
!pip install -U git+https://github.com/jianfch/stable-ts.git
!pip install transformers==4.38.2
!pip install scenedetect[opencv] --upgrade
!pip install ffmpeg-python
!pip install moviepy pysrt
!pip install pysubs2

In [ ]:
from tqdm.auto import tqdm
import stable_whisper
model = stable_whisper.load_hf_whisper('openai/whisper-large-v3-turbo',
                                       device = 'cuda')

In [ ]:
result2 = model.transcribe('Подкаст #1 Ревматоидный артрит_ мифы и реальность о ГИБП.mp3',
                          language ='russian',
                          batch_size=64)

In [ ]:
result2.to_ass('Подкаст #1 Ревматоидный артрит_ мифы и реальность о ГИБП.ass', word_level = False)
result2.to_txt('Подкаст #1 Ревматоидный артрит_ мифы и реальность о ГИБП.txt')

#### PDF

Прасинг pdf. Выделяю изобраежния, таблицы и текст. Таблицы сохранияю в csv и png. Также обрабатываю 

In [ ]:
!pip install PyMuPDF pytesseract pdf2image pandas opencv-python pillow
!pip install camelot-py

In [ ]:
import fitz  # PyMuPDF
import camelot
from PIL import Image
import os
from tqdm.auto import tqdm
import pandas as pd

def parse_pdf(filepath, output_root_dir):
    # Создаем основную директорию с именем файла (без расширения)
    filename = os.path.splitext(os.path.basename(filepath))[0]
    output_dir = os.path.join(output_root_dir, filename)
    os.makedirs(output_dir, exist_ok=True)

    # Создаем поддиректории
    images_dir = os.path.join(output_dir, "images")
    tables_dir = os.path.join(output_dir, "tables")
    texts_dir = os.path.join(output_dir, "texts")

    os.makedirs(images_dir, exist_ok=True)
    os.makedirs(tables_dir, exist_ok=True)
    os.makedirs(texts_dir, exist_ok=True)

    # Откройте PDF-файл
    doc = fitz.open(filepath)

    # Словарь для хранения таблиц, которые могут продолжаться на следующих страницах
    table_groups = {}

    # Обработка каждой страницы
    for page_num in tqdm(range(len(doc))):
        page = doc.load_page(page_num)

        # 1. Извлечение текста
        text = page.get_text()
        text_filename = f"page_{page_num}_text.txt"
        with open(os.path.join(texts_dir, text_filename), "w", encoding="utf-8") as f:
            f.write(text)

        # 2. Извлечение встроенных изображений
        img_list = page.get_images(full=True)
        for img_index, img in enumerate(img_list):
            xref = img[0]
            base_image = doc.extract_image(xref)
            img_data = base_image["image"]
            img_ext = base_image["ext"]
            img_filename = f"page_{page_num}_img_{img_index}.{img_ext}"
            img_path = os.path.join(images_dir, img_filename)
            with open(img_path, "wb") as f:
                f.write(img_data)

        # 3. Извлечение таблиц
        try:
            # Попробуем оба режима распознавания
            tables_lattice = camelot.read_pdf(
                filepath,
                pages=str(page_num + 1),
                flavor='lattice'
            )

            # Выберем тот результат, который дал больше таблиц
            tables = tables_lattice

        except Exception as e:
            print(f"Ошибка при извлечении таблиц (страница {page_num}): {str(e)}")
            tables = []

        if hasattr(tables, 'n') and tables.n > 0:
            # Рендеринг страницы в изображение (высокое разрешение)
            scale = 3  # Масштаб для четкости
            mat = fitz.Matrix(scale, scale)
            pix = page.get_pixmap(matrix=mat, alpha=False)

            # Создаем PIL изображение из всей страницы
            page_img = Image.frombytes("RGB", [pix.width, pix.height], pix.samples)

            for i, table in enumerate(tables):
                try:
                    # Получаем координаты таблицы
                    x1, y1, x2, y2 = table._bbox

                    # Преобразуем координаты в пиксели
                    page_height = page.rect.height

                    # Координаты в PDF (точки) -> пиксели
                    left = int(x1 * scale)
                    top = int((page_height - y2) * scale)  # Инверсия Y
                    right = int(x2 * scale)
                    bottom = int((page_height - y1) * scale)

                    # Проверяем корректность координат
                    if left >= 0 and top >= 0 and right <= pix.width and bottom <= pix.height and left < right and top < bottom:
                        # Вырезаем область таблицы
                        table_img = page_img.crop((left, top, right, bottom))

                        # Сохраняем таблицу как изображение
                        table_img_filename = f"page_{page_num}_table_{i}.png"
                        table_img.save(os.path.join(tables_dir, table_img_filename))

                        # Сохраняем таблицу как CSV
                        table_df = table.df
                        table_csv_filename = f"page_{page_num}_table_{i}.csv"
                        table_df.to_csv(os.path.join(tables_dir, table_csv_filename),
                                       index=False, encoding='utf-8')

                        # Проверяем, является ли таблица продолжением предыдущей
                        table_key = identify_table_continuation(table_df, table_groups, page_num, i)

                        if table_key:
                            # Добавляем к существующей группе таблиц
                            table_groups[table_key]['pages'].append(page_num)
                            table_groups[table_key]['table_indices'].append(i)
                            table_groups[table_key]['images'].append(table_img)
                            table_groups[table_key]['dataframes'].append(table_df)
                        else:
                            # Создаем новую группу таблиц
                            new_key = f"table_group_{len(table_groups)}"
                            table_groups[new_key] = {
                                'pages': [page_num],
                                'table_indices': [i],  # Добавляем индекс таблицы на странице
                                'images': [table_img],
                                'dataframes': [table_df],
                                'last_page': page_num  # Запоминаем последнюю страницу группы
                            }

                    else:
                        print(f"Некорректные координаты таблицы на странице {page_num}: "
                              f"({left}, {top}, {right}, {bottom}), размер страницы: ({pix.width}, {pix.height})")

                except Exception as e:
                    print(f"Ошибка при обработке таблицы {i} на странице {page_num}: {str(e)}")

    # Сохраняем объединенные таблицы в папку tables
    save_combined_tables(table_groups, tables_dir)

    doc.close()

def identify_table_continuation(current_df, table_groups, page_num, table_index):
    """Определяет, является ли таблица продолжением предыдущей"""
    if not table_groups:
        return None

    # Сначала проверяем таблицы с предыдущей страницы
    for key, group in table_groups.items():
        # Пропускаем группы, где последняя страница не предыдущая
        if group['last_page'] != page_num - 1:
            continue

        # Берем последнюю таблицу из группы
        last_df = group['dataframes'][-1]

        # Проверяем схожесть структуры (количество колонок)
        if len(current_df.columns) == len(last_df.columns):
            # Дополнительная проверка: схожесть заголовков или первых строк
            if columns_similar(current_df.columns, last_df.columns):
                # Обновляем последнюю страницу группы
                group['last_page'] = page_num
                return key

    # Если не нашли продолжение среди всех таблиц с предыдущей страницы,
    # проверяем таблицы с текущей страницы с меньшими индексами
    for key, group in table_groups.items():
        # Пропускаем группы, где последняя страница не текущая
        if group['last_page'] != page_num:
            continue

        # Пропускаем группы, где индекс последней таблицы не меньше текущего
        if group['table_indices'][-1] >= table_index:
            continue

        # Берем последнюю таблицу из группы
        last_df = group['dataframes'][-1]

        # Проверяем схожесть структуры (количество колонок)
        if len(current_df.columns) == len(last_df.columns):
            # Дополнительная проверка: схожесть заголовков или первых строк
            if columns_similar(current_df.columns, last_df.columns):
                # Обновляем последнюю страницу группы
                group['last_page'] = page_num
                return key

    return None

def columns_similar(cols1, cols2, threshold=0.7):
    """Проверяет схожесть заголовков колонок"""
    if len(cols1) != len(cols2):
        return False

    matches = 0
    for c1, c2 in zip(cols1, cols2):
        if str(c1).strip() == str(c2).strip():
            matches += 1

    return matches / len(cols1) >= threshold

def save_combined_tables(table_groups, tables_dir):
    """Сохраняет объединенные таблицы как единые изображения и CSV"""
    for key, group in table_groups.items():
        if len(group['images']) > 1:
            # Объединяем изображения вертикально
            combined_img = combine_images_vertically(group['images'])
            combined_img_filename = f"{key}_combined.png"
            combined_img.save(os.path.join(tables_dir, combined_img_filename))

            # Объединяем DataFrame'ы
            combined_df = pd.concat(group['dataframes'], ignore_index=True)
            combined_csv_filename = f"{key}_combined.csv"
            combined_df.to_csv(os.path.join(tables_dir, combined_csv_filename),
                             index=False, encoding='utf-8')

            print(f"Объединена таблица {key} со страниц {group['pages']}")

def combine_images_vertically(images):
    """Объединяет изображения вертикально"""
    if not images:
        return None

    # Находим максимальную ширину
    max_width = max(img.width for img in images)
    total_height = sum(img.height for img in images)

    # Создаем новое изображение
    combined = Image.new('RGB', (max_width, total_height), 'white')

    # Вставляем изображения
    current_height = 0
    for img in images:
        # Центрируем изображение по ширине, если оно уже
        x_offset = (max_width - img.width) // 2
        combined.paste(img, (x_offset, current_height))
        current_height += img.height

    return combined

/usr/local/lib/python3.11/dist-packages/pypdf/_crypt_providers/_cryptography.py:32: CryptographyDeprecationWarning: ARC4 has been moved to cryptography.hazmat.decrepit.ciphers.algorithms.ARC4 and will be removed from this module in 48.0.0.
  from cryptography.hazmat.primitives.ciphers.algorithms import AES, ARC4


In [ ]:
import os
from tqdm.auto import tqdm


for pdf_name in tqdm(os.listdir('/content/drive/MyDrive/РА')):
    parse_pdf(os.path.join('/content/drive/MyDrive/РА', pdf_name), "output")

###  Summary

#### PDF

Макслимальная длина контекста для эмбеддингов OpenAi 2048. Страницы не всегда влазят, поэтому суммаризирую тексты

In [ ]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from tqdm.auto import tqdm
from google.colab import userdata
os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI')


def generate_text_summaries(texts, summarize_texts=False, chunk_size=3):
    """
    Функция для создания суммаризации текста, обрабатывающая по chunk_size страниц за раз.

    Аргументы:
    texts: Список строк (тексты), которые нужно суммировать.
    summarize_texts: Булев флаг, указывающий, нужно ли суммировать текстовые элементы.
    chunk_size: Количество страниц для объединения перед суммаризацией.

    Возвращает:
    Список суммаризированных текстов.
    """
    if not summarize_texts:
        return texts.copy() if texts else []

    # Шаблон для запроса к модели
    prompt_text = """Тебе дается часть текста научной работы по ревматойдный артрит.
Твоя задача суммаризировать и сократить текст, оставив только важные детали.
Твоя суммаризация должна быть оптимизирована под дальнейший поиск RAG.
Текст суммаризации должен быть на русском языке, не зависимо от исходного текста.

Текст: {element}"""

    prompt = ChatPromptTemplate.from_template(prompt_text)
    model = ChatOpenAI(temperature=0.2, model='gpt-4o')
    summarize_chain = (
        {"element": lambda x: x["element"]}
        | prompt
        | model
        | StrOutputParser()
    )
    # Объединяем тексты по chunk_size страниц
    chunked_texts = []
    for i in range(0, len(texts), chunk_size):
        chunk = "\n\n".join(texts[i:i+chunk_size])
        chunked_texts.append(chunk)

    # Создаем контекст с количеством страниц для каждого чанка
    contexts = [{"element": chunk} for chunk in chunked_texts]

    # Выполняем суммаризацию параллельно
    text_summaries = summarize_chain.batch(contexts, {"max_concurrency": 10})

    return text_summaries

In [ ]:
all_docs = []
all_sum = []
for doc in tqdm(os.listdir('/content/output')):
    text_one_doc = []

    os.mkdir(f'/content/output/{doc}/summary')

    for text in os.listdir(f'/content/output/{doc}/texts'):
        with open(f'/content/output/{doc}/texts/{text}', 'r') as f:
            text_one_doc.append(f.read())

    summaries = generate_text_summaries(text_one_doc,
                                     summarize_texts=True,
                                    chunk_size=1)

    for one_summary,text in zip(summaries,
                           os.listdir(f'/content/output/{doc}/texts')):

        with open(f"/content/output/{doc}/summary/{text.replace('text', 'summary')}", 'w') as f:
            f.write(one_summary)

    all_sum.append(summaries)
    all_docs.append(text_one_doc)

### RAG

#### PDF

In [ ]:
!pip install -U langchain openai langchain-openai langchain-chroma langchain-experimental -qqq

In [ ]:
!pip install faiss-cpu

#### Добавляю документы

In [ ]:
import json
import numpy as np
import faiss
import os
from openai import OpenAI
from typing import List, Dict, Optional
from google.colab import userdata
os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI')


class VectorDatabase:
    def __init__(self, model: str = "text-embedding-3-small"):
        self.documents: List[Dict] = []
        self.client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])
        self.embedding_model = model
        self.index: Optional[faiss.Index] = None
        self.dimension = 1536  # Для text-embedding-3-small

    def _get_embedding(self, text: str) -> np.ndarray:
        """Получение эмбеддинга через OpenAI API"""
        response = self.client.embeddings.create(
            input=[text],
            model=self.embedding_model
        )
        return np.array(response.data[0].embedding, dtype='float32')

    def add_document(self, document: Dict):
        """Добавление документа в базу"""
        self.documents.append(document)
        embedding = self._get_embedding(document["summarization"])

        if self.index is None:
            self.index = faiss.IndexFlatL2(self.dimension)
            self.index.add(np.array([embedding]))
        else:
            self.index.add(np.array([embedding]))

    def add_documents_batch(self, documents: List[Dict]):
        """Пакетное добавление документов"""
        summaries = [doc["summarization"] for doc in documents]
        embeddings = [self._get_embedding(text) for text in summaries]

        self.documents.extend(documents)
        embeddings_array = np.array(embeddings, dtype='float32')

        if self.index is None:
            self.index = faiss.IndexFlatL2(self.dimension)
        self.index.add(embeddings_array)

    def search(self, query: str, k: int = 5) -> List[Dict]:
        """Поиск по сумммаризации с возвратом полных документов"""
        query_embed = self._get_embedding(query)
        distances, indices = self.index.search(np.array([query_embed]), k)

        return [self.documents[i] for i in indices[0] if i < len(self.documents)]

    def save(self, path: str):
        """Сохранение базы на диск"""
        os.makedirs(path, exist_ok=True)

        with open(os.path.join(path, "documents.json"), "w", encoding="utf-8") as f:
            json.dump(self.documents, f, ensure_ascii=False, indent=2)

        if self.index is not None:
            faiss.write_index(self.index, os.path.join(path, "index.faiss"))

    def load(self, path: str):
        """Загрузка базы с диска"""
        with open(os.path.join(path, "documents.json"), "r", encoding="utf-8") as f:
            self.documents = json.load(f)

        index_path = os.path.join(path, "index.faiss")
        if os.path.exists(index_path):
            self.index = faiss.read_index(index_path)

In [ ]:
document_content = []
for i in range(len(file_name)):
    document_content.append({'file_name':file_name[i],
                             'num_page':num_page[i],
                             'text':texts[i],
                             'summarization':summaries[i],
                             'tables':tables[i],
                             'images':images[i]})

In [ ]:
document_content[0].keys()

dict_keys(['file_name', 'num_page', 'text', 'summarization', 'tables', 'images'])

In [ ]:
len(document_content)

507

In [44]:
db = VectorDatabase()

In [45]:
db.add_documents_batch(document_content)

In [ ]:
db.save("openai_embeddings_db")


In [ ]:
db.search("Причины ревматоидного артрита", k=3)

[{'file_name': 'КР250_3',
  'num_page': 50,
  'text': 'КР250 \n89 \nРевматолог проводит осмотр суставов, кожи, других органов и систем. Затем назначает \nопределенный спектр анализов и другие методы исследования – в частности, рентген, \nУЗИ и другие. Диагноз ставится на основании степени поражения суставов, обнаружения \nэрозий суставных поверхностей при рентгенологическом исследовании, выявления в \nсыворотке крови ревматоидного фактора (особого белка, который появляется у \nбольшинства пациентов). В крови повышается СОЭ, уровень фибриногена, С-реактивного \nбелка. \n \nК сожалению, абсолютно специфических (патогномоничных) признаков ревматоидного \nартрита не существует, однако при обнаружении ревматоидного фактора и антител к \nцитруллинированным \nбелкам \nвероятность \nревматоидного \nартрита \nсушественно \nповышается. Поэтому диагностика всегда осуществляется по комплексу данных, \nполученных при обследовании, а не с помощью какого-либо одного анализа или \nисследования. \n \nУ

### Загрузка бота

In [ ]:
!pip install nest_asyncio python-telegram-bot pandas

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sys
sys.path.append('/content/drive/MyDrive/Итоговый проект/src')

In [11]:
from simple_bot import start_tg

In [ ]:
from google.colab import userdata
import os
TOKEN_TG = userdata.get('AIbolit_TG_TOKEN')
OPENAI_API_KEY = userdata.get('OPENAI')
os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY
start_tg(TOKEN_TG=TOKEN_TG, OPENAI_API_KEY=OPENAI_API_KEY)

In [ ]:
rag = RAGSystem(api_key = userdata.get('OPENAI'))
answer = rag.query_ai("Какие симптомы у ревматоидного артрита?")
print(answer)

Ревматоидный артрит (РА) — это хроническое аутоиммунное заболевание, которое преимущественно поражает суставы. Основные симптомы включают:

1. **Боль и припухлость суставов:** Наиболее часто поражаются мелкие суставы кистей и стоп. Боль и отек обычно проявляются симметрично.

2. **Утренняя скованность:** Пациенты часто ощущают скованность в суставах по утрам, которая может длиться более 30 минут.

3. **Покраснение и тепло в области суставов:** Воспаленные суставы могут быть теплыми и красными.

4. **Синдром Шегрена:** Может сопровождаться сухостью глаз и полости рта из-за воспаления желез.

5. **Уставляемость и легкая лихорадка:** Пациенты могут испытывать общую усталость, субфебрильную температуру тела, потерю аппетита и похудание.

6. **Ревматоидные узелки:** Утолщение кожи над костями, обычно на локтях.

7. **Другие системные проявления:** РА также может поражать легкие (интерстициальное заболевание легких), сердце и кровеносные сосуды, увеличивая риск кардиоваскулярных заболеваний 